## 一. 前半部分

我们遵循与流匹配版本**完全相同**的数学逻辑，将 DiffusionNFT 的核心推导完整移植到基于得分匹配（Score Matching）的扩散模型上，覆盖 **得分空间（Score Space）**、**噪声预测空间（$\epsilon$-Space）** 和 **原始数据预测空间（$x_0$-Space）**。所有推导均从概率空间的线性切片出发，通过得分函数的凸组合，再经由 Tweedie 公式等仿射桥接，消去由时间步引入的额外项，最终在每个空间中都得到干净的三点共线结构和统一的外推目标。

---

## 1. 概率空间的线性切片与时间传播（完全通用）

设旧策略（预训练模型）的干净图像分布为 $p_{\text{old}}(x_0|c)$，奖励函数 $r(x_0,c) \in [0,1]$。定义平均最优性：
$$
\gamma = \mathbb{E}_{p_{\text{old}}}[r(x_0,c)].
$$
用 $r$ 对旧分布进行重加权，得到正、负策略：
$$
p^+(x_0) = \frac{r(x_0)}{\gamma} p_{\text{old}}(x_0), \qquad
p^-(x_0) = \frac{1-r(x_0)}{1-\gamma} p_{\text{old}}(x_0).
$$
显见它们满足全概率恒等式：
$$
p_{\text{old}}(x_0) = \gamma\, p^+(x_0) + (1-\gamma)\, p^-(x_0). \tag{1.1}
$$
考虑 VP SDE 或一般高斯扩散的前向加噪过程，其转移核为：
$$
p_{t|0}(x_t|x_0) = \mathcal{N}(x_t; \alpha_t x_0, \sigma_t^2 \mathbf{I}).
$$
对于任意分布 $p(x_0)$，其在时刻 $t$ 的边缘分布为 $p_t(x_t) = \int p_{t|0}(x_t|x_0) p(x_0) dx_0$。由于积分是线性算子，将 (1.1) 两边同时卷积 $p_{t|0}$，即得：
$$
p_t^{\text{old}}(x_t) = \gamma\, p_t^+(x_t) + (1-\gamma)\, p_t^-(x_t). \tag{1.2}
$$
这正是所有后续推导的起点。

---

## 2. 得分函数空间的凸组合

对 (1.2) 式两边关于空间变量 $x_t$ 求梯度 $\nabla_{x_t}$：
$$
\nabla p_t^{\text{old}} = \gamma \nabla p_t^+ + (1-\gamma) \nabla p_t^-.
$$
利用恒等式 $\nabla p = p \cdot \nabla \log p$，得：
$$
p_t^{\text{old}} \cdot s_{\text{old}} = \gamma\, p_t^+ \cdot s^+ + (1-\gamma)\, p_t^- \cdot s^-,
$$
其中 $s(x_t) = \nabla_{x_t} \log p_t(x_t)$ 为得分函数。两边除以 $p_t^{\text{old}}$，并定义动态权重系数：
$$
\alpha(x_t) := \frac{\gamma\, p_t^+(x_t)}{p_t^{\text{old}}(x_t)} = \mathbb{E}_{p_{\text{old}}(x_0|x_t)}[r(x_0)], \tag{2.1}
$$
则利用 (1.2) 可自动得到第二项系数为 $1-\alpha(x_t)$。于是我们得到**得分函数的严格凸组合**：
$$
\boxed{s_{\text{old}}(x_t) = \alpha(x_t)\, s^+(x_t) + \big(1-\alpha(x_t)\big)\, s^-(x_t)}. \tag{2.2}
$$
在得分匹配框架中，这已经是核心空间，无需再做空间转换，后续的噪声空间和 $x_0$ 空间均由它导出。

---

## 3. 噪声预测空间（$\epsilon$-Space）的仿射桥接

在实际的扩散模型（如 DDPM、Stable Diffusion）中，网络通常预测添加的噪声 $\epsilon$。VP SDE 的加噪重参数化为：
$$
x_t = \alpha_t x_0 + \sigma_t \epsilon, \quad \epsilon \sim \mathcal{N}(0,\mathbf{I}).
$$
根据 Tweedie 公式，得分函数与最优噪声预测器 $\epsilon(x_t)$ 之间存在严格的线性关系：
$$
s(x_t) = \nabla_{x_t} \log p_t(x_t) = -\frac{\epsilon(x_t)}{\sigma_t}. \tag{3.1}
$$
将 (3.1) 分别代入旧、正、负策略的得分函数，并代入得分凸组合 (2.2)：
$$
-\frac{\epsilon_{\text{old}}(x_t)}{\sigma_t} = \alpha(x_t)\left(-\frac{\epsilon^+(x_t)}{\sigma_t}\right) + (1-\alpha(x_t))\left(-\frac{\epsilon^-(x_t)}{\sigma_t}\right).
$$
两边同乘以 $-\sigma_t$（消去时间系数），立即得到**噪声空间的线性恒等式**：
$$
\boxed{\epsilon_{\text{old}}(x_t) = \alpha(x_t)\, \epsilon^+(x_t) + \big(1-\alpha(x_t)\big)\, \epsilon^-(x_t)}. \tag{3.2}
$$
这一步骤与流匹配中消去 $A_t x_t$ 异曲同工：时间依赖的缩放因子 $\sigma_t$ 被直接消除，恢复了纯净的凸组合结构。

---

## 4. 原始数据预测空间（$x_0$-Space）的仿射桥接

另一种常见参数化是直接预测干净图像 $x_0$，即 $\hat{x}_0(x_t) = \mathbb{E}[x_0|x_t]$。再次利用 Tweedie 公式，得分函数与 $\hat{x}_0$ 的仿射关系为：
$$
s(x_t) = \frac{\alpha_t}{\sigma_t^2}\left( \hat{x}_0(x_t) - \frac{1}{\alpha_t} x_t \right). \tag{4.1}
$$
将 (4.1) 代入得分凸组合 (2.2)：
$$
\frac{\alpha_t}{\sigma_t^2}\left( \hat{x}_0^{\text{old}} - \frac{1}{\alpha_t} x_t \right)
= \alpha(x_t) \frac{\alpha_t}{\sigma_t^2}\left( \hat{x}_0^+ - \frac{1}{\alpha_t} x_t \right)
+ (1-\alpha(x_t)) \frac{\alpha_t}{\sigma_t^2}\left( \hat{x}_0^- - \frac{1}{\alpha_t} x_t \right).
$$
两边同除以公共因子 $\frac{\alpha_t}{\sigma_t^2}$：
$$
\hat{x}_0^{\text{old}} - \frac{1}{\alpha_t} x_t
= \alpha(x_t)\left( \hat{x}_0^+ - \frac{1}{\alpha_t} x_t \right)
+ (1-\alpha(x_t))\left( \hat{x}_0^- - \frac{1}{\alpha_t} x_t \right).
$$
展开右侧含 $x_t$ 的项：
$$
\alpha(x_t) \cdot \frac{1}{\alpha_t} x_t + (1-\alpha(x_t)) \cdot \frac{1}{\alpha_t} x_t = \frac{1}{\alpha_t} x_t,
$$
恰好与左侧的 $-\frac{1}{\alpha_t} x_t$ 抵消。由此得到极其简洁的**$x_0$ 预测空间的线性恒等式**：
$$
\boxed{\hat{x}_0^{\text{old}}(x_t) = \alpha(x_t)\, \hat{x}_0^+(x_t) + \big(1-\alpha(x_t)\big)\, \hat{x}_0^-(x_t)}. \tag{4.2}
$$
物理意义：旧模型对干净图像的预测，在任何时刻 $t$ 都是正、负样本预测的动态凸组合，当前带噪状态 $x_t$ 的影响被完美消去。

---

## 5. 各空间中的三点共线与强化引导方向 $\Delta$

从凸组合恒等式出发，通过完全相同的移项手法，我们可在每个空间中得到“向正靠拢”与“向负远离”的共线关系。

### 5.1 得分空间（Score Space）
由 (2.2) 进行移项：
$$
s_{\text{old}} - s^- = \alpha (s^+ - s^-), \qquad
s^+ - s_{\text{old}} = (1-\alpha)(s^+ - s^-).
$$
消去公共基准 $s^+ - s^-$，得：
$$
\alpha(s^+ - s_{\text{old}}) = (1-\alpha)(s_{\text{old}} - s^-).
$$
定义**得分强化引导方向**：
$$
\boxed{\Delta_s(x_t, c, t) := \alpha(x_t)\big[s^+(x_t) - s_{\text{old}}(x_t)\big] = \big(1-\alpha(x_t)\big)\big[s_{\text{old}}(x_t) - s^-(x_t)\big]}. \tag{5.1}
$$

### 5.2 噪声空间（$\epsilon$-Space）
由 (3.2) 同样操作：
$$
\alpha(\epsilon^+ - \epsilon_{\text{old}}) = (1-\alpha)(\epsilon_{\text{old}} - \epsilon^-).
$$
定义**噪声强化引导方向**：
$$
\boxed{\Delta_\epsilon(x_t, c, t) := \alpha(x_t)\big[\epsilon^+(x_t) - \epsilon_{\text{old}}(x_t)\big] = \big(1-\alpha(x_t)\big)\big[\epsilon_{\text{old}}(x_t) - \epsilon^-(x_t)\big]}. \tag{5.2}
$$
由于 $s = -\epsilon/\sigma_t$，显然有 $\Delta_s = -\frac{1}{\sigma_t}\Delta_\epsilon$，方向完全共线。

### 5.3 $x_0$ 预测空间（$x_0$-Space）
由 (4.2) 同理：
$$
\alpha(\hat{x}_0^+ - \hat{x}_0^{\text{old}}) = (1-\alpha)(\hat{x}_0^{\text{old}} - \hat{x}_0^-).
$$
定义 **$x_0$ 强化引导方向**：
$$
\boxed{\Delta_{x_0}(x_t, c, t) := \alpha(x_t)\big[\hat{x}_0^+(x_t) - \hat{x}_0^{\text{old}}(x_t)\big] = \big(1-\alpha(x_t)\big)\big[\hat{x}_0^{\text{old}}(x_t) - \hat{x}_0^-(x_t)\big]}. \tag{5.3}
$$

---

## 6. 统一的目标策略与外推损失函数

在每个空间中，我们都得到了一条贯穿“负 → 旧 → 正”的直线。引入引导强度参数 $\beta > 0$，便可定义任意外推幅度的目标策略。

### 6.1 得分空间的目标 $s^*$
$$
s^*(x_t, c, t) := s_{\text{old}}(x_t) + \frac{1}{\beta}\Delta_s(x_t, c, t). \tag{6.1}
$$
将 $\Delta_s$ 的定义代入，可得其显式的凸组合或外推形式：
$$
s^* = \left(1 - \frac{\alpha}{\beta}\right)s_{\text{old}} + \frac{\alpha}{\beta}s^+.
$$
当 $\beta = \alpha$ 时，$s^* = s^+$（收敛至正策略）；当 $\beta < \alpha$ 时，沿 $\Delta_s$ 方向越过 $s^+$ 进一步外推。

### 6.2 噪声空间的目标 $\epsilon^*$
$$
\epsilon^*(x_t, c, t) := \epsilon_{\text{old}}(x_t) + \frac{1}{\beta}\Delta_\epsilon(x_t, c, t). \tag{6.2}
$$
对应地：
$$
\epsilon^* = \left(1 - \frac{\alpha}{\beta}\right)\epsilon_{\text{old}} + \frac{\alpha}{\beta}\epsilon^+.
$$

### 6.3 $x_0$ 空间的目标 $x_0^*$
$$
x_0^*(x_t, c, t) := \hat{x}_0^{\text{old}}(x_t) + \frac{1}{\beta}\Delta_{x_0}(x_t, c, t). \tag{6.3}
$$
展开为：
$$
x_0^* = \left(1 - \frac{\alpha}{\beta}\right)\hat{x}_0^{\text{old}} + \frac{\alpha}{\beta}\hat{x}_0^+.
$$

### 6.4 对应的训练损失函数
要让单个可训练模型（例如 $s_\theta, \epsilon_\theta, \hat{x}_0^\theta$）去逼近上述目标策略，只需采用与 DiffusionNFT 完全相同的隐式参数化技巧。以噪声预测为例，定义：
$$
\epsilon_\theta^+(x_t) := (1-\beta)\epsilon_{\text{old}}(x_t) + \beta \epsilon_\theta(x_t), \\
\epsilon_\theta^-(x_t) := (1+\beta)\epsilon_{\text{old}}(x_t) - \beta \epsilon_\theta(x_t).
$$
然后使用奖励加权的复合 MSE 损失：
$$
\mathcal{L}_\epsilon(\theta) = \mathbb{E}_{c,x_0,t,\epsilon}\Big[ r\|\epsilon_\theta^+ - \epsilon\|_2^2 + (1-r)\|\epsilon_\theta^- - \epsilon\|_2^2 \Big]. \tag{6.4}
$$
类似地，得分空间可写为：
$$
\mathcal{L}_s(\theta) = \mathbb{E}\Big[ r\|s_\theta^+ - \nabla\log p(x_t|x_0)\|_2^2 + (1-r)\|s_\theta^- - \nabla\log p(x_t|x_0)\|_2^2 \Big],
$$
$x_0$ 预测空间则为：
$$
\mathcal{L}_{x_0}(\theta) = \mathbb{E}\Big[ r\|\hat{x}_{0,\theta}^+ - x_0\|_2^2 + (1-r)\|\hat{x}_{0,\theta}^- - x_0\|_2^2 \Big].
$$
在无限数据与完美优化下，这些损失的最优解恰好就是前面的 $s^*, \epsilon^*, x_0^*$，从而实现等价于强化学习的策略改进。

---

## 7. 总结：四大表征空间的统一

| 表征空间 | 线性组合形式 | 消除的关键时间项 | 物理意义 |
|---------|-------------|----------------|---------|
| 概率空间 | $p_t^{\text{old}} = \gamma p_t^+ + (1-\gamma)p_t^-$ | 无（源头） | 全概率线性切片 |
| 得分空间 | $s_{\text{old}} = \alpha s^+ + (1-\alpha)s^-$ | 无（直接得到） | 对数梯度的凸组合 |
| 噪声空间 ($\epsilon$) | $\epsilon_{\text{old}} = \alpha \epsilon^+ + (1-\alpha)\epsilon^-$ | 消去 $\sigma_t$（分母） | 噪声预测的凸组合 |
| $x_0$ 预测空间 | $\hat{x}_0^{\text{old}} = \alpha \hat{x}_0^+ + (1-\alpha)\hat{x}_0^-$ | 消去 $\frac{1}{\alpha_t}x_t$（漂移项） | 干净图像的凸组合 |

无论在哪种参数化下，“**正策略与负策略之间的改进方向都是同一条直线**”，并且可以通过调整 $\beta$ 在该直线上自由滑动——从保守逼近到激进外推。这证明了 DiffusionNFT 的核心思想是**表征无关的**，可以无缝适配 Score Matching、VP SDE、$x_0$ 预测等主流扩散模型训练范式，且所有理论保证（共线性、最优解形式、外推灵活性）全部保留。

## 二. 后半部分

DiffusionNFT 的真实灵感路径是 **“从 LLM 的 NFT 复合损失出发，在扩散模型的噪声预测空间中用单个模型表达正负策略，再通过求导得到最优解，最后反向包装出共线定理”**。

我们用一个统一的符号 $h$ 来代表扩散模型可预测的任意目标，一次性完成对噪声预测（$\epsilon$）、原始数据预测（$x_0$）和得分匹配（$\nabla\log p_t$）三种情形的推导。

---

## 1. 通用隐式正负预测器（单模型双分身）

设预训练扩散模型在任意参数化下的预测头为 $h_{\text{old}}$（冻结），当前可训练模型为 $h_\theta$。引入超参数 $\beta > 0$，定义：

$$
\boxed{
\begin{aligned}
h_\theta^+(x_t) &:= (1-\beta)\,h_{\text{old}}(x_t) + \beta\,h_\theta(x_t) \quad \text{(隐式正策略)} \\[4pt]
h_\theta^-(x_t) &:= (1+\beta)\,h_{\text{old}}(x_t) - \beta\,h_\theta(x_t) \quad \text{(隐式负策略)}
\end{aligned}
}
$$

在不同参数化下，$h$ 的具体含义：
- 噪声预测：$h = \epsilon$；
- $x_0$ 预测：$h = x_0$；
- 得分匹配：$h = \nabla_{x_t}\log p_t(x_t|x_0)$。

**代数性质**：
- $h_\theta^+ + h_\theta^- = 2h_{\text{old}}$ （中心对称）
- $h_\theta^+ - h_\theta^- = 2\beta(h_\theta - h_{\text{old}})$ （差异完全由 $h_\theta$ 控制）
- 当 $h_\theta = h_{\text{old}}$ 时，两者都退化为旧模型。

---

## 2. 复合回归损失（直接迁移 LLM NFT 形式）

对每张生成图像 $x_0$，由奖励模型给出其最优性概率 $r(x_0)\in[0,1]$。训练时采样时间 $t$，加噪得 $x_t$，并获取目标值 $h(x_t|x_0)$。损失函数为：

$$
\boxed{
\mathcal{L}(\theta) = \mathbb{E}_{c,\,x_0\sim\pi_{\text{old}},\,t}\Big[
r\,\|h_\theta^+(x_t) - h(x_t|x_0)\|^2 + (1-r)\,\|h_\theta^-(x_t) - h(x_t|x_0)\|^2
\Big]
} \tag{1}
$$

- $r \approx 1$ 时主要训练正分身去拟合 $h$；
- $r \approx 0$ 时主要训练负分身去拟合 $h$；
- 两个分身共享同一个 $h_\theta$，因此梯度同时影响两者。

---

## 3. 逐点优化与极值必要条件

在无限数据、无限容量假设下，可在每个时空点 $(x_t,c,t)$ 独立优化 $h_\theta(x_t)$。以下固定该点，所有量均为该点的向量，期望 $\mathbb{E}$ 为给定 $x_t$ 下对 $(x_0,r,h)$ 的条件期望。

将 $h_\theta^+,h_\theta^-$ 代入损失，对 $h_\theta$ 求梯度并令其为零：

$$
\nabla_{h_\theta}\mathcal{L}_{\text{local}} = 2\,\mathbb{E}\Big[
r\beta\big((1-\beta)h_{\text{old}} + \beta h_\theta - h\big)
- (1-r)\beta\big((1+\beta)h_{\text{old}} - \beta h_\theta - h\big)
\Big] = \mathbf{0}
$$

除以 $2\beta$（$\beta>0$）得必要条件：

$$
\mathbb{E}\Big[
r\big((1-\beta)h_{\text{old}} + \beta h_\theta - h\big)
- (1-r)\big((1+\beta)h_{\text{old}} - \beta h_\theta - h\big)
\Big] = \mathbf{0} \tag{2}
$$

---

## 4. 展开并归类

展开 (2)：

$$
\mathbb{E}\Big[
r(1-\beta)h_{\text{old}} + r\beta h_\theta - r h \\
- (1-r)(1+\beta)h_{\text{old}} + (1-r)\beta h_\theta + (1-r)h
\Big] = \mathbf{0}
$$

合并同类项系数：

- $h_\theta$ 系数：$r\beta + (1-r)\beta = \beta$
- $h_{\text{old}}$ 系数：$r(1-\beta) - (1-r)(1+\beta) = 2r - 1 - \beta$
- $h$ 系数：$-r + (1-r) = 1 - 2r$

于是 (2) 等价于：

$$
\mathbb{E}\Big[ \beta h_\theta + (2r - 1 - \beta)h_{\text{old}} + (1-2r)h \Big] = \mathbf{0} \tag{3}
$$

---

## 5. 用条件期望和关键量代入

定义该点 $x_t$ 处的后验期望：

- $\alpha := \mathbb{E}[r \mid x_t]$ （动态权重）
- $h_{\text{old}} := \mathbb{E}[h \mid x_t]$ （旧策略的最优预测）
- $h^+ := \mathbb{E}_{\pi^+}[h \mid x_t]$ （正策略的最优预测）
- $h^- := \mathbb{E}_{\pi^-}[h \mid x_t]$ （负策略的最优预测）

利用重要性加权换底，对所有参数化均成立：

$$
\mathbb{E}[r\,h \mid x_t] = \alpha\,h^+, \qquad
\mathbb{E}[(1-r)\,h \mid x_t] = (1-\alpha)\,h^-
$$

对 (3) 式取条件期望（$h_\theta, h_{\text{old}}$ 可提出期望算子）：

$$
\beta h_\theta + \big(2\alpha - 1 - \beta\big)h_{\text{old}} + \big(\mathbb{E}[h] - 2\mathbb{E}[rh]\big) = \mathbf{0}
$$

代入 $\mathbb{E}[h] = h_{\text{old}}$ 和 $\mathbb{E}[rh] = \alpha h^+$：

$$
\beta h_\theta + (2\alpha - 1 - \beta)h_{\text{old}} + h_{\text{old}} - 2\alpha h^+ = \mathbf{0}
$$

---

## 6. 合并 $h_{\text{old}}$ 项并解出 $h_\theta$

$$
\beta h_\theta + \big(2\alpha - 1 - \beta + 1\big)h_{\text{old}} - 2\alpha h^+ = \mathbf{0}
$$

$$
\beta h_\theta + (2\alpha - \beta)h_{\text{old}} - 2\alpha h^+ = \mathbf{0}
$$

移项：

$$
\beta h_\theta = 2\alpha h^+ - (2\alpha - \beta)h_{\text{old}}
$$

右边重新组合：

$$
\begin{aligned}
\beta h_\theta &= 2\alpha h^+ - 2\alpha h_{\text{old}} + \beta h_{\text{old}} \\
&= 2\alpha (h^+ - h_{\text{old}}) + \beta h_{\text{old}}
\end{aligned}
$$

两边减去 $\beta h_{\text{old}}$：

$$
\beta (h_\theta - h_{\text{old}}) = 2\alpha (h^+ - h_{\text{old}})
$$

因为 $\beta>0$，同除 $\beta$ 得：

$$
\boxed{h_\theta^* = h_{\text{old}} + \frac{2}{\beta}\,\alpha(h^+ - h_{\text{old}})} \tag{4}
$$

---

## 7. 定义通用强化引导方向 $\Delta$，得最终形式

由对称性可知在所有参数化空间中都有共线关系：

$$
\alpha(h^+ - h_{\text{old}}) = (1-\alpha)(h_{\text{old}} - h^-)
$$

定义**通用强化引导方向**：

$$
\boxed{\Delta(x_t, c, t) := \alpha(h^+ - h_{\text{old}}) = (1-\alpha)(h_{\text{old}} - h^-)} \tag{5}
$$

代入 (4) 即得与原论文公式 (6) 完全对齐的最优解形式：

$$
\boxed{h_\theta^*(x_t, c, t) = h_{\text{old}}(x_t, c, t) + \frac{2}{\beta}\,\Delta(x_t, c, t)} \tag{6}
$$

---

## 8. 三种具体参数化的对应关系

只需将 $h$ 和 $h_{\text{target}}$ 明确，所有结果直接适用：


| 参数化 | 预测目标 $h(x_t\mid x_0)$ | 最优预测 $h_{\text{old}}, h^+, h^-$ | 最优解形式 |
|--------|----------------------------|-------------------------------------|-----------|
| 噪声预测 | $\epsilon$ | 各策略下对 $\epsilon$ 的条件期望 | $\epsilon_\theta^* = \epsilon_{\text{old}} + \frac{2}{\beta}\Delta_\epsilon$ |
| $x_0$ 预测 | $x_0$ | 各策略下对 $x_0$ 的条件期望 | $\hat{x}_{0,\theta}^* = \hat{x}_{0,\text{old}} + \frac{2}{\beta}\Delta_{x_0}$ |
| 得分匹配 | $\nabla_{x_t}\!\log p_t(x_t\mid x_0)$ | 各策略下对得分的条件期望 | $s_\theta^* = s_{\text{old}} + \frac{2}{\beta}\Delta_s$ |


---

## 9. 反推共线定理与外推灵活性

由 (5) 可知，$\Delta$ 同时代表“从旧策略向正策略靠拢”和“从负策略向旧策略靠拢”的方向，即两者共线。调节 $\beta$ 即可在该直线上自由外推：

- $\beta = \alpha$ 时，$h_\theta^* = h^+$，恰为纯净正策略；
- $\beta < \alpha$ 时，步长更大，越过正策略进一步强化；
- $\beta \to \infty$ 时，$h_\theta^* \to h_{\text{old}}$，不做改进。

---

**总结**：DiffusionNFT 的核心机制完全表征无关。用通用符号 $h$ 统一代表扩散模型的预测目标后，隐式参数化、复合 MSE 损失和最优解结构在所有参数化下完全一致。只需在具体实现时把 $h$ 替换为 $\epsilon$、$x_0$ 或 $\nabla\log p_t$，即可无缝推广到噪声预测、$x_0$ 预测和得分匹配等主流扩散模型训练范式。

整个推导展现了 DiffusionNFT 在得分匹配 / 噪声预测框架下的完整迁移：**仅需将速度场替换为噪声预测器，隐式参数化和复合 MSE 损失的形式完全不变，最优解结构也完全一致**。这证明了该算法对扩散模型参数化方式的普适性。